# 01 — Data Profiling & Quality Assessment

**Owner:** Person 1 — rrrafaa. **Source:** World Bank Global Findex Indonesia, 2024.

Run on **Colab or local Jupyter**. Download official source privately. Do not publish individual respondent data or identifiers. Missing data may be due to survey routing, not zeros.


## A. Load original ZIP, check checksum and shape

In Colab, upload the original ZIP when prompted. Locally, keep the ZIP in `data/raw/`.


In [ ]:
from pathlib import Path
import hashlib, zipfile, io, sys, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
REPO="Indonesia-Financial-Inclusion-Digital-Finance-Analytics"
ZIP="IDN_2024_FINDEX_v02_M_CSV.zip"
CSV="Findex_Microdata_2025_updateIndonesia.csv"
SHA="fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
root=next((p for p in [Path.cwd(),*Path.cwd().parents,Path("/content")/REPO] if (p/"data/raw/README.md").exists()),None)
if root is None and "google.colab" in sys.modules:
    root=Path("/content")/REPO
    subprocess.run(["git","clone","--depth","1","https://github.com/rrrafaa/"+REPO+".git",str(root)],check=True)
if root is None: raise FileNotFoundError("Open Jupyter in the cloned repository folder")
raw=root/"data/raw"; out=root/"data/processed"
raw.mkdir(parents=True,exist_ok=True); out.mkdir(parents=True,exist_ok=True)
path=next((p for p in [raw/ZIP,Path("/content")/ZIP,Path.cwd()/ZIP] if p.exists()),None)
if path is None and "google.colab" in sys.modules:
    from google.colab import files
    uploaded=files.upload()
    if ZIP not in uploaded: raise FileNotFoundError("Upload original "+ZIP)
    path=raw/ZIP;path.write_bytes(uploaded[ZIP])
if path is None: raise FileNotFoundError("Put the original ZIP in data/raw/"+ZIP)
if hashlib.sha256(path.read_bytes()).hexdigest()!=SHA: raise ValueError("Unexpected ZIP version; checksum differs from source README")
with zipfile.ZipFile(path) as z:
    df=pd.read_csv(io.BytesIO(z.read(CSV)),low_memory=False)
assert df.shape==(1068,199) and set(df.year.unique())=={2024}
assert df.wgt.notna().all() and df.wgt.gt(0).all()
print("Source validated:",df.shape,"|",path)



## B. Inspect schema and missingness


In [ ]:
print("Rows",len(df),"Columns",len(df.columns))
print(df.dtypes.value_counts())
profile=pd.DataFrame({"variable":df.columns,"dtype":df.dtypes.astype(str).values,"non_missing":df.notna().sum().values,"missing":df.isna().sum().values,"missing_pct":(100*df.isna().mean()).round(2).values,"nunique":df.nunique().values})
display(profile.sort_values("missing_pct",ascending=False).head(30))
profile.to_csv(out/"person1_schema_profile.csv",index=False)
print("Columns:",", ".join(df.columns))


## C. Duplicates, respondent identifiers, consistency checks


In [ ]:
print("Duplicate rows:",df.duplicated().sum())
print("Duplicate respondent IDs:",df.wpid_random.duplicated().sum())
for v in ["female","educ","inc_q","urbanicity","account","account_fin","account_mob","anydigpayment","internet_use"]:
    print(v,df[v].value_counts(dropna=False).to_dict())
assert df.wpid_random.is_unique
assert df.age.between(15,120).all()
assert df.female.isin([1,2]).all()
assert df.account.isin([0,1]).all()
assert df.inc_q.isin(range(1,6)).all()
assert df.urbanicity.isin([1,2]).all()


## D. Understand missing values and survey skip logic

`fin11a`–`fin11f` were asked of a subpopulation. Do not replace their missing values by 0.


In [ ]:
barriers=[f"fin11{x}" for x in "abcdef"]
print(df[barriers].isna().sum())
display(pd.crosstab(df.account_fin,df.fin11a.isna(),normalize="index").round(3))
assert df.loc[df.account_fin.eq(1),"fin11a"].isna().all()


## E. Weight diagnostics

The Kish effective sample size is a *rough approximation*, not a substitute for complex-survey confidence intervals.


In [ ]:
w=df.wgt
print(w.describe().round(3))
print("Kish effective N:",round(w.sum()**2/w.pow(2).sum(),1))
fig,ax=plt.subplots(figsize=(7,4));ax.hist(w,bins=30)
ax.set(title="Distribution of survey weights",xlabel="wgt",ylabel="Respondents");plt.show()


## F. Export safe metadata reports locally (ignored by Git)

Review aggregate small cells before any public upload.


In [ ]:
columns=["female","educ","inc_q","urbanicity","account","account_fin","account_mob","anydigpayment","saved","borrowed"]
frequency=pd.concat([df[x].value_counts(dropna=False).rename_axis("value").reset_index(name="count").assign(variable=x) for x in columns],ignore_index=True)
display(frequency)
frequency.to_csv(out/"person1_value_frequencies.csv",index=False)
checks=pd.DataFrame({"check":["rows","columns","duplicate_rows","duplicate_ids","weight_nulls","educ_missing"],"value":[len(df),df.shape[1],int(df.duplicated().sum()),int(df.wpid_random.duplicated().sum()),int(df.wgt.isna().sum()),int(df.educ.isna().sum())]})
checks.to_csv(out/"person1_quality_checks.csv",index=False)
display(checks)
print("Saved locally under:",out)


## Handoff checklist

- [ ] Review profiling, missingness and category-code discrepancies.
- [ ] Record the survey weighting decision.
- [ ] Do not publish or commit the raw ZIP/CSV.
- [ ] Continue with `04_eda_inclusion.ipynb` for the Person 1 assigned analysis.
